# Export XGBoost Model for Live Trading

Loads `data/latest_features.jsonl`, trains XGBoost with optimal features
from `data/optimal_features_xgb.json` (generated by notebook 14),
fits isotonic calibrator, exports model + scaler + features + calibrator to `models/`.


In [ ]:
import json
import random
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score, brier_score_loss, classification_report
from sklearn.preprocessing import StandardScaler

random.seed(42)
np.random.seed(42)

FEATURES_PATH = Path("../data/latest_features.jsonl")
XGB_FEATURES_PATH = Path("../data/optimal_features_xgb.json")
MODELS_DIR = Path("../models")

## 1. Load data and optimal features

In [ ]:
rows = []
with open(FEATURES_PATH) as f:
    for line in f:
        rows.append(json.loads(line))

df = pd.DataFrame(rows)
df["target"] = (df["outcome"] == "UP").astype(int)

# Load optimal XGB features and hyperparameters
with open(XGB_FEATURES_PATH) as _f:
    xgb_config = json.load(_f)
    feat_cols = xgb_config["features"]
    hyperparams = xgb_config.get("hyperparameters", {})

print(f"Using {len(feat_cols)} optimal XGB features from {XGB_FEATURES_PATH.name}")
print(f"Hyperparameters: {hyperparams}")

df[feat_cols] = df[feat_cols].fillna(0.0)

print(f"Loaded {len(df):,} rows, {df['candle_id'].nunique()} candles")
print(f"UP rate: {df['target'].mean() * 100:.1f}%")

## 2. Train/val split for validation metrics

In [ ]:
candle_ids = df["candle_id"].unique()
split_idx = int(len(candle_ids) * 0.8)
train_ids = set(candle_ids[:split_idx])
val_ids = set(candle_ids[split_idx:])

df_train = df[df["candle_id"].isin(train_ids)]
df_val = df[df["candle_id"].isin(val_ids)]

scaler = StandardScaler()
X_train = scaler.fit_transform(df_train[feat_cols].values)
y_train = df_train["target"].values
X_val = scaler.transform(df_val[feat_cols].values)
y_val = df_val["target"].values

model = xgb.XGBClassifier(
    random_state=42,
    eval_metric="logloss",
    n_jobs=-1,
    **hyperparams,
)
model.fit(X_train, y_train)

# Raw validation
raw_probs = model.predict_proba(X_val)[:, 1]
raw_preds = (raw_probs >= 0.5).astype(int)
raw_acc = accuracy_score(y_val, raw_preds)

# Calibrated validation
cal_model = CalibratedClassifierCV(model, method="isotonic", cv=3)
cal_model.fit(X_train, y_train)
cal_probs = cal_model.predict_proba(X_val)[:, 1]
cal_preds = (cal_probs >= 0.5).astype(int)
cal_acc = accuracy_score(y_val, cal_preds)

print(f"Validation ({df_val['candle_id'].nunique()} candles, {len(df_val):,} rows):")
print(f"  Raw:        accuracy={raw_acc * 100:.1f}%  Brier={brier_score_loss(y_val, raw_probs):.4f}")
print(f"  Calibrated: accuracy={cal_acc * 100:.1f}%  Brier={brier_score_loss(y_val, cal_probs):.4f}")
print()
print(classification_report(y_val, cal_preds, target_names=["DOWN", "UP"]))

## 3. Train on ALL data and export

In [ ]:
# Retrain scaler and model on ALL data
scaler = StandardScaler()
X_all = scaler.fit_transform(df[feat_cols].values)
y_all = df["target"].values

model = xgb.XGBClassifier(
    random_state=42,
    eval_metric="logloss",
    n_jobs=-1,
    **hyperparams,
)
model.fit(X_all, y_all)

# Calibrate on ALL data (cv=3 internal split)
cal_model = CalibratedClassifierCV(model, method="isotonic", cv=3)
cal_model.fit(X_all, y_all)

# Export
joblib.dump(model, MODELS_DIR / "xgb_v1.joblib")
joblib.dump(scaler, MODELS_DIR / "xgb_scaler_v1.joblib")
joblib.dump(feat_cols, MODELS_DIR / "xgb_feature_cols_v1.joblib")
joblib.dump(cal_model, MODELS_DIR / "xgb_calibrator_v1.joblib")

print(f"Saved to {MODELS_DIR}/:")
for name in ["xgb_v1.joblib", "xgb_scaler_v1.joblib", "xgb_feature_cols_v1.joblib", "xgb_calibrator_v1.joblib"]:
    path = MODELS_DIR / name
    size = path.stat().st_size
    print(f"  {name}: {size:,} bytes")

## 4. Verify exported model

In [ ]:
# Reload and verify
loaded_model = joblib.load(MODELS_DIR / "xgb_v1.joblib")
loaded_scaler = joblib.load(MODELS_DIR / "xgb_scaler_v1.joblib")
loaded_features = joblib.load(MODELS_DIR / "xgb_feature_cols_v1.joblib")
loaded_cal = joblib.load(MODELS_DIR / "xgb_calibrator_v1.joblib")

# Predict on a sample
sample = df[feat_cols].iloc[:10].values
X_sample = loaded_scaler.transform(sample)
raw_probs = loaded_model.predict_proba(X_sample)[:, 1]
cal_probs = loaded_cal.predict_proba(X_sample)[:, 1]

print(f"Features: {len(loaded_features)}")
print("Sample predictions (raw vs calibrated):")
for i in range(10):
    print(f"  Row {i}: raw={raw_probs[i]:.3f}  cal={cal_probs[i]:.3f}  truth={df['target'].iloc[i]}")

print("\nModel ready for live trading.")

## 5. Conclusion

Exported XGBoost model with:
- Optimal features from `data/optimal_features_xgb.json`
- Tuned hyperparameters from notebook 14
- Isotonic probability calibration

Files in `models/`:
- `xgb_v1.joblib` — trained XGBClassifier
- `xgb_scaler_v1.joblib` — StandardScaler
- `xgb_feature_cols_v1.joblib` — feature column list
- `xgb_calibrator_v1.joblib` — CalibratedClassifierCV (isotonic)

To use in the bot, create a new adapter that loads these files (similar to `JoblibPredictor`).
